# Baseline модели для временных рядов Favorita

В ноутбуке строим и сравниваем baseline-модели:
1. Naive
2. SeasonalNaive
3. AutoETS
4. AutoTheta

Используем:
- метрику соревнования $NWRMSLE$;
- стратегию валидации `expanding window`;
- разбиение по каждому отдельному ряду `(store_nbr, item_nbr)`;
- случайный поднабор: 10 `store_nbr` и 100 `item_nbr`.

In [39]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install catboost statsforecast

In [3]:
import random
from pathlib import Path
from typing import Sequence, Tuple, Union

import catboost as cb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, Naive, SeasonalNaive

pio.renderers.default = "vscode"

In [4]:
data_dir = Path('/content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)')
train_csv = data_dir / 'train.csv'

In [5]:
stores_df = pd.read_csv(train_csv, usecols=['store_nbr'])

stores_df['store_nbr'].unique()

array([25,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 23, 24, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39,
       40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 54, 36, 53, 20, 29, 21,
       42, 22, 52])

In [6]:
np.random.seed(4207)
random_10_stores = np.random.choice(stores_df['store_nbr'].unique(), size=10)

random_10_stores

array([ 3, 28,  5, 35, 45, 53, 42, 54,  9, 18])

In [7]:
items_df = pd.read_csv(train_csv, usecols=['item_nbr'])

items_df['item_nbr'].unique()

array([ 103665,  105574,  105575, ..., 2126944, 2123839, 2011451])

In [8]:
np.random.seed(4207)
random_100_items = np.random.choice(items_df['item_nbr'].unique(), size=100)

random_100_items

array([ 457688,  876223,  949243, 1447815, 1958179, 1660193,  115847,
       1036689,  310644, 1463510,  805310, 1958231,  857331, 1658994,
        165553,  857331,  215468,  523054, 2016619,  953562, 1464094,
       1926106,  111397, 2026683, 1972443,  638977, 2018612,  977007,
        257847,  108862, 1696003, 1938600,  574898,  856687,  105857,
        849136,  413557, 1146797, 1237005,  371437, 1109389, 2046789,
       1459058, 1945572, 1017349, 1239862, 1005463, 1965358, 1489662,
       1695875, 1162382, 2010755, 1464070, 2011259, 2081050, 2011155,
        407608, 1464074, 1909453, 2053614, 2010793,  828630, 1239813,
        376194, 1105211, 1444615, 2010370, 1963684, 1726002,  116279,
       1393033, 2008747, 1473474, 1695875, 1457335, 1463857, 1456997,
       1176562, 1089820, 2047495,  796394,  205387, 1464210, 1167754,
       1428331,  559493, 2057762, 1940454,  795611,  956011, 1473483,
       1913257, 2058895,  261700,  165727,  977007, 1975602, 1464188,
       1370564, 1460

In [9]:
from tqdm import tqdm

In [10]:
# Читаем train.csv чанками без агрегирования, чтобы сохранить уровень (date, store_nbr, item_nbr).
chunksize = 20_000_000

parts = []
for chunk in tqdm(pd.read_csv(
    train_csv,
    usecols=['date', 'store_nbr', 'item_nbr', 'unit_sales'],
    parse_dates=['date'],
    chunksize=chunksize,
)):
    # В данных встречаются отрицательные значения возвратов; клипуем их в 0.
    chunk = chunk[(chunk['store_nbr'].isin(random_10_stores)) & (chunk['item_nbr'].isin(random_100_items))]
    chunk['unit_sales'] = chunk['unit_sales'].clip(lower=0)
    chunk = chunk.sort_values(['date'])
    parts.append(chunk)

if not parts:
    raise ValueError('Не удалось прочитать train.csv: файл пустой или не считан.')

train_panel = pd.concat(parts, ignore_index=True)
print('concated')
# train_panel = train_panel.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)
# print("sorted")

print(train_panel.head())
print(train_panel.tail())
print(f'Строк в train_panel: {len(train_panel):,}')
print(f'Уникальных пар (store_nbr, item_nbr): {train_panel[["store_nbr", "item_nbr"]].drop_duplicates().shape[0]:,}')

# Для временного анализа ниже берём пару store/item с максимальной длиной истории.
pair_counts = train_panel.groupby(['store_nbr', 'item_nbr'], as_index=False).size()
target_pair = pair_counts.loc[pair_counts['size'].idxmax(), ['store_nbr', 'item_nbr']]
target_store = int(target_pair['store_nbr'])
target_item = int(target_pair['item_nbr'])

ts_daily = (
    train_panel[
        (train_panel['store_nbr'] == target_store)
        & (train_panel['item_nbr'] == target_item)
    ]
    .set_index('date')['unit_sales']
    .sort_index()
)
ts_daily.name = 'unit_sales'

print(f'Ряд для анализа: store_nbr={target_store}, item_nbr={target_item}, длина={len(ts_daily)}')
print(ts_daily.head())
print(ts_daily.tail())

0it [00:00, ?it/s]/tmp/ipykernel_81897/2249459972.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

1it [00:16, 16.93s/it]/tmp/ipykernel_81897/2249459972.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

2it [00:35, 18.03s/it]/tmp/ipykernel_81897/2249459972.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gui

concated
        date  store_nbr  item_nbr  unit_sales
0 2013-01-02          3    105857      36.000
1 2013-01-02         28    876223       7.000
2 2013-01-02         28    956011       4.000
3 2013-01-02         28   1017349       1.439
4 2013-01-02         28   1036689      13.000
             date  store_nbr  item_nbr  unit_sales
587861 2017-08-15          9   1464210        14.0
587862 2017-08-15          9   1464094        11.0
587863 2017-08-15          9   1464074        10.0
587864 2017-08-15          9   2010370         1.0
587865 2017-08-15         54   2053614         4.0
Строк в train_panel: 587,866
Уникальных пар (store_nbr, item_nbr): 830
Ряд для анализа: store_nbr=5, item_nbr=1036689, длина=1679
date
2013-01-02    38.0
2013-01-03    15.0
2013-01-04    10.0
2013-01-05    13.0
2013-01-06    20.0
Name: unit_sales, dtype: float64
date
2017-08-11    12.0
2017-08-12    15.0
2017-08-13    18.0
2017-08-14    17.0
2017-08-15    12.0
Name: unit_sales, dtype: float64


In [25]:
!pip install -q catboost statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 12.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 4.6 MB/s eta 0:00:00


In [26]:
import importlib
import sys

sys.path.insert(0, '/content')

import modules
from modules.validation import expanding_window_validation
from modules.models import StatsforecastModel


from statsforecast.models import AutoETS, AutoTheta, Naive, SeasonalNaive

In [27]:
import modules
importlib.reload(modules)
importlib.reload(modules.validation)
import modules
from modules.validation import expanding_window_validation

In [13]:
HISTORY = 28
HORIZON = 14
START_TRAIN_SIZE = 12 * 7  # 12 недель
STEP_SIZE = 7  # 1 неделя

SEASON_LENGTH = 7  # Недельная сезонность
FREQ = "D"  # Частота временного ряда, нужна для StatsForecast

In [14]:
df = train_panel

In [15]:
df['date'] = pd.to_datetime(df['date'])

In [16]:
df.columns

Index(['date', 'store_nbr', 'item_nbr', 'unit_sales'], dtype='object')

In [25]:
!mv /content/validation.py /content/modules/validation.py

In [18]:
df['sensor_id'] = df['store_nbr'].astype(str) + '_' + df['item_nbr'].astype(str)

In [19]:
df['timestamp'] = pd.to_datetime(df['date'])

In [20]:
df['y'] = df['unit_sales']

In [21]:
df.dtypes

,0
date,datetime64[ns]
store_nbr,int64
item_nbr,int64
unit_sales,float64
sensor_id,object
timestamp,datetime64[ns]
y,float64


In [24]:
min(df['date'].unique())

Timestamp('2013-01-02 00:00:00')

In [ ]:
train_end_idx = + pd.DateOffset(hours=start_train_size)


In [28]:
all_results = {}

for model, model_name in [
    (StatsforecastModel(Naive(), FREQ, HORIZON), "Naive"),
    (StatsforecastModel(SeasonalNaive(season_length=SEASON_LENGTH), FREQ, HORIZON), "SeasonalNaive"),
    (StatsforecastModel(AutoETS(season_length=SEASON_LENGTH), FREQ, HORIZON), "AutoETS"),
    (StatsforecastModel(AutoTheta(season_length=SEASON_LENGTH), FREQ, HORIZON), "AutoTheta"),
]:
    print(f"Evaluating model: {model_name}")
    results_df = expanding_window_validation(
        data=df[['sensor_id', 'timestamp', 'y']],
        model=model,
        horizon=HORIZON,
        history=HISTORY,
        start_train_size=START_TRAIN_SIZE,
        step_size=STEP_SIZE,
        id_col='sensor_id',
        timestamp_col="timestamp",
        value_col="y",
    )

    all_results[model_name] = results_df


Evaluating model: Naive
Nan count in predictions 0
(2487,)
(2487, 3)
(2487, 3)
(2487,)
(3416,)


ValueError: All arrays must be of the same length

In [ ]:
all_results_df = (
    pd.concat(all_results.values(), keys=all_results.keys(), names=["model", "row_id"])
    .reset_index()
    .drop(columns=["row_id"])
)
all_results_df

In [ ]:
def plot_results_expanding_cv(
    df: pd.DataFrame,
    forecast_df: pd.DataFrame,
    id_column: str = "sensor_id",
    time_column: str = "timestamp",
    forecast_column: str = "predicted_value",
    target_column_df: str = "value",
    target_column_forecast_df: str = "true_value",
    fold_column_forecast_df: str = "fold",
    model_column_forecast_df: str = "model",
    num_samples_to_plot: int = 3,
    seed: int = 42,
):
    """Визуализирует expanding window CV.
    Args:
        df: Датафрейм с реальными значениями.
        forecast_df: Датафрейм с прогнозами и информацией о фолдах.
        id_column: Название колонки с идентификатором временного ряда.
        time_column: Название колонки с временной меткой.
        forecast_column: Название колонки с прогнозами.
        target_column_df: Название колонки с реальными значениями в df.
        target_column_forecast_df: Название колонки с реальными значениями в forecast_df.
        fold_column_forecast_df: Название колонки с фолдами в forecast_df.
        model_column_forecast_df: Название колонки с именами моделей в forecast_df.
        num_samples_to_plot: Количество случайных рядов для визуализации.
        seed: random seed для воспроизводимости.

    """
    colors = {
        "history_true": "grey",
        "forecast_true": "blue",
        "forecast_models": px.colors.qualitative.Plotly
    }
    
    random.seed(seed)
    sampled_ids = random.sample(
        forecast_df[id_column].dropna().unique().tolist(), num_samples_to_plot
    )

    num_folds = forecast_df[fold_column_forecast_df].nunique()
    for series_id in sampled_ids:
        current_df = df[df[id_column] == series_id]
        current_forecast_df = forecast_df[forecast_df[id_column] == series_id]
        meta_row = sensor_meta[sensor_meta['sensor_id'] == series_id]
        meta_suffix = ''
        if not meta_row.empty:
            meta_suffix = (
                f" (store_nbr={int(meta_row['store_nbr'].iloc[0])}, "
                f"item_nbr={int(meta_row['item_nbr'].iloc[0])})"
            )

        fig = make_subplots(
            rows=num_folds,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.12,
            subplot_titles=[f"fold={f}" for f in range(num_folds)],
        )

        for fold_idx in range(num_folds):
            current_fold_forecast_df = current_forecast_df[
                current_forecast_df[fold_column_forecast_df] == fold_idx
            ]
            test_start_timestamp = current_fold_forecast_df[time_column].min()

            history = current_df[current_df[time_column] < test_start_timestamp]

            fig.add_trace(
                go.Scatter(
                    x=history[time_column],
                    y=history[target_column_df],
                    mode="lines",
                    name="History (train+val)",
                    legendgroup="history",
                    line=dict(color=colors["history_true"]),
                    showlegend=(fold_idx == 0),
                ),
                row=fold_idx + 1,
                col=1,
            )
            for i, model_name in enumerate(sorted(
                current_fold_forecast_df[model_column_forecast_df].dropna().unique().tolist()
            )):                
                mdf = current_fold_forecast_df[
                    current_fold_forecast_df[model_column_forecast_df] == model_name
                ]
                if i == 0:
                    fig.add_trace(
                        go.Scatter(
                            x=mdf[time_column],
                            y=mdf[target_column_forecast_df],
                            mode="lines",
                            name="True (test)",
                            legendgroup="true_test",
                            line=dict(color=colors["forecast_true"]),
                            showlegend=(fold_idx == 0),
                        ),
                        row=fold_idx + 1,
                        col=1,
                    )
                fig.add_trace(
                    go.Scatter(
                        x=mdf[time_column],
                        y=mdf[forecast_column],
                        mode="lines",
                        line=dict(dash="dash", color=colors["forecast_models"][i % len(colors["forecast_models"])]),
                        name=str(model_name),
                        legendgroup=str(model_name),
                        showlegend=(fold_idx == 0),
                    ),
                    row=fold_idx + 1,
                    col=1,
                )
        fig.update_layout(
            title=f"Expanding CV: sensor_id={series_id}{meta_suffix}",
        )
        fig.show()

In [ ]:
plot_results_expanding_cv(df, all_results_df, num_samples_to_plot=1, seed=None)